# Dataset EDA

이 노트북은 다음 항목을 정리합니다.
- 폴더(split)별 샘플 수 / 파일 수
- 이미지(front/top) 크기 통계
- 영상(simulation.mp4) 길이/프레임 수/FPS 통계
- 클래스(label) 비율

In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

try:
    import cv2
except ImportError:
    cv2 = None

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

# /src 실행을 기본으로 하되, 다른 위치에서도 동작하도록 데이터 경로를 탐색
candidate_dirs = [Path('../data'), Path('data')]
DATA_DIR = next((d for d in candidate_dirs if d.exists()), None)
assert DATA_DIR is not None, f'데이터 폴더를 찾을 수 없습니다. checked={ [str(d.resolve()) for d in candidate_dirs] }'

assert DATA_DIR.exists(), f'데이터 폴더를 찾을 수 없습니다: {DATA_DIR.resolve()}'

train_df = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8-sig')
dev_df = pd.read_csv(DATA_DIR / 'dev.csv', encoding='utf-8-sig')

print('train.csv shape:', train_df.shape)
print('dev.csv shape:  ', dev_df.shape)

train.csv shape: (1000, 2)
dev.csv shape:   (100, 2)


In [4]:
# 1) split별 샘플/파일 개수
def summarize_split(split_name: str) -> dict:
    split_dir = DATA_DIR / split_name
    sample_dirs = [p for p in split_dir.iterdir() if p.is_dir()]

    n_png = len(list(split_dir.glob('*/*.png')))
    n_mp4 = len(list(split_dir.glob('*/*.mp4')))

    return {
        'split': split_name,
        'sample_dirs': len(sample_dirs),
        'png_files': n_png,
        'mp4_files': n_mp4,
        'expected_png_per_sample(2)': n_png / len(sample_dirs) if sample_dirs else np.nan,
    }

split_summary = pd.DataFrame([summarize_split(s) for s in ['train', 'dev', 'test']])
split_summary

,split,sample_dirs,png_files,mp4_files,expected_png_per_sample(2)
0,train,1000,2000,1000,2.0
1,dev,100,200,0,2.0
2,test,1000,2000,0,2.0


In [5]:
# 2) 이미지 크기 통계(front/top)
image_paths = list(DATA_DIR.glob('train/*/*.png')) + list(DATA_DIR.glob('dev/*/*.png')) + list(DATA_DIR.glob('test/*/*.png'))

img_rows = []
for p in tqdm(image_paths, desc='Reading images'):
    split = p.parts[-3]
    sample_id = p.parts[-2]
    view = p.stem  # front/top
    with Image.open(p) as im:
        w, h = im.size
        mode = im.mode

    img_rows.append({
        'split': split,
        'sample_id': sample_id,
        'view': view,
        'width': w,
        'height': h,
        'mode': mode,
        'size_str': f'{w}x{h}',
    })

img_df = pd.DataFrame(img_rows)
print('Total images:', len(img_df))
img_df.head()

Total images: 4200


,split,sample_id,view,width,height,mode,size_str
0,train,TRAIN_0006,top,384,384,RGB,384x384
1,train,TRAIN_0006,front,384,384,RGB,384x384
2,train,TRAIN_0132,top,384,384,RGB,384x384
3,train,TRAIN_0132,front,384,384,RGB,384x384
4,train,TRAIN_0301,top,384,384,RGB,384x384


In [6]:
img_size_stats = (
    img_df.groupby(['split', 'view'])[['width', 'height']]
    .agg(['count', 'min', 'max', 'mean', 'std'])
    .round(2)
)
img_size_stats

width                       height                      
            count  min  max   mean  std  count  min  max   mean  std
split view                                                          
dev   front   100  384  384  384.0  0.0    100  384  384  384.0  0.0
      top     100  384  384  384.0  0.0    100  384  384  384.0  0.0
test  front  1000  384  384  384.0  0.0   1000  384  384  384.0  0.0
      top    1000  384  384  384.0  0.0   1000  384  384  384.0  0.0
train front  1000  384  384  384.0  0.0   1000  384  384  384.0  0.0
      top    1000  384  384  384.0  0.0   1000  384  384  384.0  0.0

In [7]:
size_dist = (
    img_df.groupby(['split', 'view', 'size_str'])
    .size()
    .reset_index(name='count')
    .sort_values(['split', 'view', 'count'], ascending=[True, True, False])
)

# 각 split/view에서 가장 많은 해상도 상위 5개
top_size_dist = (
    size_dist.groupby(['split', 'view'])
    .head(5)
    .reset_index(drop=True)
)
top_size_dist

,split,view,size_str,count
0,dev,front,384x384,100
1,dev,top,384x384,100
2,test,front,384x384,1000
3,test,top,384x384,1000
4,train,front,384x384,1000
5,train,top,384x384,1000


In [8]:
# 3) 영상 길이/프레임 수 통계(train의 simulation.mp4)
video_paths = list(DATA_DIR.glob('train/*/simulation.mp4')) + list(DATA_DIR.glob('dev/*/simulation.mp4')) + list(DATA_DIR.glob('test/*/simulation.mp4'))

if cv2 is None:
    print('opencv-python(cv2)이 없어 비디오 메타데이터를 읽을 수 없습니다.')
    video_df = pd.DataFrame()
else:
    video_rows = []
    for vp in tqdm(video_paths, desc='Reading videos'):
        split = vp.parts[-3]
        sample_id = vp.parts[-2]

        cap = cv2.VideoCapture(str(vp))
        if not cap.isOpened():
            video_rows.append({
                'split': split,
                'sample_id': sample_id,
                'video_path': str(vp),
                'opened': False,
                'frame_count': np.nan,
                'fps': np.nan,
                'duration_sec': np.nan,
                'width': np.nan,
                'height': np.nan,
            })
            cap.release()
            continue

        frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
        height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)

        frame_count = int(frame_count) if frame_count and frame_count > 0 else np.nan
        fps = float(fps) if fps and fps > 0 else np.nan
        duration_sec = (frame_count / fps) if pd.notna(frame_count) and pd.notna(fps) else np.nan

        video_rows.append({
            'split': split,
            'sample_id': sample_id,
            'video_path': str(vp),
            'opened': True,
            'frame_count': frame_count,
            'fps': fps,
            'duration_sec': duration_sec,
            'width': int(width) if width else np.nan,
            'height': int(height) if height else np.nan,
        })
        cap.release()

    video_df = pd.DataFrame(video_rows)

print('Total videos found:', len(video_paths))
video_df.head() if len(video_paths) else 'No videos found'

Total videos found: 1000


,split,sample_id,video_path,opened,frame_count,fps,duration_sec,width,height
0,train,TRAIN_0006,../data/train/TRAIN_0006/simulation.mp4,True,300,30.0,10.0,384,384
1,train,TRAIN_0132,../data/train/TRAIN_0132/simulation.mp4,True,300,30.0,10.0,384,384
2,train,TRAIN_0301,../data/train/TRAIN_0301/simulation.mp4,True,300,30.0,10.0,384,384
3,train,TRAIN_0573,../data/train/TRAIN_0573/simulation.mp4,True,300,30.0,10.0,384,384
4,train,TRAIN_0754,../data/train/TRAIN_0754/simulation.mp4,True,300,30.0,10.0,384,384


In [9]:
if not video_df.empty:
    video_summary = (
        video_df.groupby('split')[['frame_count', 'fps', 'duration_sec', 'width', 'height']]
        .agg(['count', 'min', 'max', 'mean', 'std'])
        .round(3)
    )
    display(video_summary)

    unresolved = video_df[~video_df['opened']]
    print('Failed to open videos:', len(unresolved))
else:
    print('비디오 통계를 계산할 데이터가 없습니다.')

frame_count                         fps                        duration_sec                        width                        \
            count  min  max   mean  std count   min   max  mean  std        count   min   max  mean  std count  min  max   mean  std   
split                                                                                                                                  
train        1000  300  300  300.0  0.0  1000  30.0  30.0  30.0  0.0         1000  10.0  10.0  10.0  0.0  1000  384  384  384.0  0.0   

      height                        
       count  min  max   mean  std  
split                               
train   1000  384  384  384.0  0.0

Failed to open videos: 0


In [10]:
# 4) 클래스(label) 비율
def label_ratio(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    out = (
        df['label']
        .value_counts(dropna=False)
        .rename_axis('label')
        .reset_index(name='count')
    )
    out['split'] = split_name
    out['ratio'] = out['count'] / out['count'].sum()
    out['ratio_pct'] = (out['ratio'] * 100).round(2)
    return out[['split', 'label', 'count', 'ratio', 'ratio_pct']]

label_stats = pd.concat([
    label_ratio(train_df, 'train'),
    label_ratio(dev_df, 'dev')
], ignore_index=True)

label_stats.sort_values(['split', 'label'])

,split,label,count,ratio,ratio_pct
3,dev,stable,48,0.48,48.0
2,dev,unstable,52,0.52,52.0
1,train,stable,500,0.50,50.0
0,train,unstable,500,0.50,50.0


In [11]:
# 5) 핵심 요약표
core_summary = split_summary.copy()

# 각 split의 대표 이미지 해상도(최빈값)
mode_size = (
    size_dist.sort_values(['split', 'count'], ascending=[True, False])
    .groupby('split')
    .first()[['size_str', 'count']]
    .rename(columns={'size_str': 'most_common_img_size', 'count': 'most_common_img_size_count'})
)
core_summary = core_summary.merge(mode_size, left_on='split', right_index=True, how='left')

if not video_df.empty:
    v = video_df.groupby('split').agg(
        video_count=('video_path', 'count'),
        mean_duration_sec=('duration_sec', 'mean'),
        mean_frame_count=('frame_count', 'mean'),
        mean_fps=('fps', 'mean'),
    ).round(3)
    core_summary = core_summary.merge(v, left_on='split', right_index=True, how='left')

core_summary

,split,sample_dirs,png_files,mp4_files,expected_png_per_sample(2),most_common_img_size,most_common_img_size_count,video_count,mean_duration_sec,mean_frame_count,mean_fps
0,train,1000,2000,1000,2.0,384x384,1000,1000.0,10.0,300.0,30.0
1,dev,100,200,0,2.0,384x384,100,NaN,NaN,NaN,NaN
2,test,1000,2000,0,2.0,384x384,1000,NaN,NaN,NaN,NaN
